<a href="https://colab.research.google.com/github/nahom-d54/AMHARIC-VOICE-CLONING/blob/main/tts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Thu Sep 10 12:40:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:",
      round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
      "GB")

PyTorch: 2.11.0+cu128
CUDA: 12.8
GPU: Tesla T4
VRAM: 14.56 GB


In [4]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/amharic_tts")

for directory in [
    "checkpoints",
    "datasets",
    "samples",
    "logs",
    "configs",
    "manifests",
    "audio",
    "tokens",
    "final_model",
]:
    (PROJECT / directory).mkdir(parents=True, exist_ok=True)

print(PROJECT)

/content/drive/MyDrive/amharic_tts


In [6]:
!python --version
!pip --version

Python 3.13.15
pip 24.1.2 from /usr/local/lib/python3.13/dist-packages/pip (python 3.13)


In [7]:
!pip install -q -U omnivoice datasets soundfile librosa accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.5/168.5 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 8.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


In [8]:
!git clone https://github.com/k2-fsa/OmniVoice.git

Cloning into 'OmniVoice'...
remote: Enumerating objects: 558, done.
remote: Counting objects: 100% (241/241), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 558 (delta 165), reused 139 (delta 139), pack-reused 317 (from 3)
Receiving objects: 100% (558/558), 1.37 MiB | 15.99 MiB/s, done.
Resolving deltas: 100% (298/298), done.


In [9]:
%cd /content/OmniVoice

/content/OmniVoice


In [10]:
!pip install -q -e .

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for omnivoice (pyproject.toml) ... done


In [11]:
import omnivoice

print("OmniVoice imported successfully")

OmniVoice imported successfully


/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


In [12]:
!pip install -q soundfile

In [13]:
import torch
import soundfile as sf

from omnivoice import OmniVoice, OmniVoiceGenerationConfig

MODEL_ID = "african-low-resource/omnivoice-amharic"

model = OmniVoice.from_pretrained(
    MODEL_ID,
    device_map="cuda:0",
    dtype=torch.float16,
)

print("Model loaded")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'FetchError: Could not fetch resource at https://colab.research.google.com/userdata/get?authuser=0&notebookid=1_04QmJf_xuKNlQd3zL0Xfc8vfUqVR1XT&key=HF_TOKEN: 401  '.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

Model loaded


In [14]:
text = "ሰላም፣ ይህ የአማርኛ የድምፅ ሙከራ ነው።"

audio = model.generate(
    text=text,
    language="Amharic",
    generation_config=OmniVoiceGenerationConfig(
        num_step=32,
        guidance_scale=2.0,
    ),
)

sf.write(
    "/content/drive/MyDrive/amharic_tts/samples/base_amharic.wav",
    audio[0],
    24000,
)

In [15]:
text = "ሰላም፣ ይህ የአማርኛ የድምፅ ሙከራ ነው።"

audio = model.generate(
    text=text,
    language="Amharic",
    generation_config=OmniVoiceGenerationConfig(
        num_step=32,
        guidance_scale=2.0,
    ),
)

sf.write(
    "/content/drive/MyDrive/amharic_tts/samples/base_amharic.wav",
    audio[0],
    24000,
)

In [16]:
from IPython.display import Audio, display

display(
    Audio(
        "/content/drive/MyDrive/amharic_tts/samples/base_amharic.wav"
    )
)

In [17]:
from google.colab import files

uploaded = files.upload()

Saving output.wav to output.wav


In [18]:
REFERENCE = next(iter(uploaded.keys()))

prompt = model.create_voice_clone_prompt(
    ref_audio=REFERENCE,
    ref_text="መጽሐፍ ማንበብ በጣም እወዳለሁ። በተለይ ስለ ታሪክና ስለ ቴክኖሎጂ የተጻፉ መጻሕፍትን።",
)

In [19]:
text = "ዛሬ በኢትዮጵያ በጣም ጥሩ ቀን ነው።"

audio = model.generate(
    text=text,
    language="Amharic",
    voice_clone_prompt=prompt,
)

sf.write(
    "/content/drive/MyDrive/amharic_tts/samples/base_clone.wav",
    audio[0],
    24000,
)

In [20]:
display(
    Audio(
        "/content/drive/MyDrive/amharic_tts/samples/base_clone.wav"
    )
)

In [21]:
from datasets import load_dataset

dataset = load_dataset(
    "snapwre/amharic-speech"
)

dataset

README.md:   0%|          | 0.00/10.3k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

data/train-00000-of-00028.parquet: reconstructing file:   0%|          |  0.00B /  191MB            

data/train-00000-of-00028.parquet: downloading bytes:           |  0.00B            

...

data/test-00003-of-00004.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/13792 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1526 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1548 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['audio', 'clip_id', 'sentence', 'speaker_id', 'language', 'duration_s', 'speech_s', 'speech_start_s', 'speech_end_s', 'lufs', 'gender', 'age_band', 'region', 'sample_rate', 'up_votes', 'down_votes'],
        num_rows: 13792
    })
    validation: Dataset({
        features: ['audio', 'clip_id', 'sentence', 'speaker_id', 'language', 'duration_s', 'speech_s', 'speech_start_s', 'speech_end_s', 'lufs', 'gender', 'age_band', 'region', 'sample_rate', 'up_votes', 'down_votes'],
        num_rows: 1526
    })
    test: Dataset({
        features: ['audio', 'clip_id', 'sentence', 'speaker_id', 'language', 'duration_s', 'speech_s', 'speech_start_s', 'speech_end_s', 'lufs', 'gender', 'age_band', 'region', 'sample_rate', 'up_votes', 'down_votes'],
        num_rows: 1548
    })
})

In [22]:
for split in dataset:
    print("\nSPLIT:", split)
    print("Rows:", len(dataset[split]))
    print("Columns:")
    print(dataset[split].column_names)


SPLIT: train
Rows: 13792
Columns:
['audio', 'clip_id', 'sentence', 'speaker_id', 'language', 'duration_s', 'speech_s', 'speech_start_s', 'speech_end_s', 'lufs', 'gender', 'age_band', 'region', 'sample_rate', 'up_votes', 'down_votes']

SPLIT: validation
Rows: 1526
Columns:
['audio', 'clip_id', 'sentence', 'speaker_id', 'language', 'duration_s', 'speech_s', 'speech_start_s', 'speech_end_s', 'lufs', 'gender', 'age_band', 'region', 'sample_rate', 'up_votes', 'down_votes']

SPLIT: test
Rows: 1548
Columns:
['audio', 'clip_id', 'sentence', 'speaker_id', 'language', 'duration_s', 'speech_s', 'speech_start_s', 'speech_end_s', 'lufs', 'gender', 'age_band', 'region', 'sample_rate', 'up_votes', 'down_votes']


# Calculate Dataset Statistics

In [23]:
sample = dataset["train"][0]

for key, value in sample.items():
    if key == "audio":
        print(
            key,
            {
                "sampling_rate": value["sampling_rate"],
                "shape": value["array"].shape,
            }
        )
    else:
        print(key, value)

audio {'sampling_rate': 16000, 'shape': (173016,)}
clip_id clip_031090be2c68
sentence የምክር ቤቱ ጸሐፊ ሼህ ሁሴን በሽር በበኩላቸው፤ በዓሉን እስልምና በሚያዘው መሰረት በአብሮነትና በፍቅር ልናሳልፈው ይገባል ብለዋል፡፡
speaker_id spk_6674d883613f
language am
duration_s 10.812999725341797
speech_s 9.173999786376953
speech_start_s 0.7360000014305115
speech_end_s 10.812999725341797
lufs -28.399999618530273
gender male
age_band None
region None
sample_rate 16000
up_votes 3
down_votes 0


In [24]:
from collections import Counter

for split in dataset:
    counts = Counter(dataset[split]["speaker_id"])

    print(
        f"\n{split}: "
        f"{len(counts)} speakers"
    )

    print(
        "Min clips:",
        min(counts.values())
    )

    print(
        "Max clips:",
        max(counts.values())
    )

    print(
        "Average clips:",
        sum(counts.values()) / len(counts)
    )


train: 360 speakers
Min clips: 1
Max clips: 152
Average clips: 38.31111111111111

validation: 28 speakers
Min clips: 5
Max clips: 140
Average clips: 54.5

test: 105 speakers
Min clips: 1
Max clips: 138
Average clips: 14.742857142857142


# checking dupllicates

In [26]:
for split in dataset:
    texts = dataset[split]["sentence"]

    duplicates = len(texts) - len(set(texts))

    print(
        split,
        "duplicate transcripts:",
        duplicates
    )

train duplicate transcripts: 1339
validation duplicate transcripts: 17
test duplicate transcripts: 22


# Trying to normalize it

In [27]:
import unicodedata
import re

def normalize_amharic(text):
    if text is None:
        return ""

    text = str(text)

    # Unicode canonical normalization
    text = unicodedata.normalize("NFC", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    # Remove leading/trailing whitespace
    text = text.strip()

    return text

In [28]:
examples = [
    "ሰላም     እንዴት ነህ?",
    "  ይህ  የአማርኛ   ሙከራ ነው። "
]

for x in examples:
    print(normalize_amharic(x))

ሰላም እንዴት ነህ?
ይህ የአማርኛ ሙከራ ነው።


## Filter Bad Audio

- MIN_DURATION = 1.5
- MAX_DURATION = 30.0
- MIN_SPEECH = 0.8

In [30]:
MIN_DURATION = 1.5
MAX_DURATION = 30.0
MIN_SPEECH = 0.8
def valid_sample(row):
    text = normalize_amharic(row["sentence"])
    if not text:
        return False

    if row["duration_s"] < MIN_DURATION:
        return False

    if row["duration_s"] > MAX_DURATION:
        return False

    if row["speech_s"] < MIN_SPEECH:
        return False

    return True

In [31]:
for split in dataset:
    valid = sum(
        valid_sample(dataset[split][i])
        for i in range(len(dataset[split]))
    )

    print(
        split,
        valid,
        "/",
        len(dataset[split])
    )

train 13792 / 13792
validation 1526 / 1526
test 1548 / 1548


## OmniVoice's training pipeline expects JSONL entries of the form:
```
{
  "id": "...",
  "audio_path": "...",
  "text": "...",
  "language_id": "am"
}
```

In [32]:
import json
import os
import soundfile as sf
from pathlib import Path

AUDIO_DIR = PROJECT / "audio"

for split in ["train", "validation", "test"]:
    (AUDIO_DIR / split).mkdir(
        parents=True,
        exist_ok=True
    )

In [33]:
def export_split(split_name):
    ds = dataset[split_name]

    manifest_path = (
        PROJECT
        / "manifests"
        / f"{split_name}.jsonl"
    )

    with open(
        manifest_path,
        "w",
        encoding="utf-8"
    ) as manifest:

        for i in range(len(ds)):

            row = ds[i]

            if not valid_sample(row):
                continue

            text = normalize_amharic(
                row["sentence"]
            )

            clip_id = row["clip_id"]

            audio_path = (
                AUDIO_DIR
                / split_name
                / f"{clip_id}.wav"
            )

            audio = row["audio"]

            sf.write(
                audio_path,
                audio["array"],
                audio["sampling_rate"]
            )

            item = {
                "id": clip_id,
                "audio_path": str(
                    audio_path.resolve()
                ),
                "text": text,
                "language_id": "am",
            }

            manifest.write(
                json.dumps(
                    item,
                    ensure_ascii=False
                ) + "\n"
            )

    return manifest_path

In [ ]:
train_manifest = export_split("train")
val_manifest = export_split("validation")
test_manifest = export_split("test")

print(train_manifest)
print(val_manifest)
print(test_manifest)

save:

- checkpoints
- tokens
- models

back to Drive.

In [ ]:
!mkdir -p /content/amharic_tts

In [ ]:
!cp -r /content/drive/MyDrive/amharic_tts/audio /content/amharic_tts/

In [ ]:
!cp -r /content/drive/MyDrive/amharic_tts/manifests /content/amharic_tts/

# Proof-of-Concept Dataset

- check if omni voice can improve the amharic quality with this corpus

In [ ]:
from collections import defaultdict

speaker_to_rows = defaultdict(list)

for i in range(len(dataset["train"])):
    row = dataset["train"][i]

    speaker_to_rows[
        row["speaker_id"]
    ].append(i)